# 1. Los datos

Cada fila es una publicación de MercadoLibre y hay que predecir si el producto es nuevo (`new`) o usado (`used`). Se mide con accuracy.

En este notebook se pasa del JSON crudo a una tabla. El código está en la sección de carga de `datos.py`.

In [1]:
import sys
import warnings

sys.path.insert(0, "..")  # Los módulos están en la raíz del proyecto
warnings.filterwarnings("ignore")
%matplotlib inline

import json

import pandas as pd

import datos

## Un item crudo

Cada línea del archivo es un JSON con campos anidados como fotos, envío, medios de pago, ficha técnica.

In [2]:
with open(datos.TRAIN_RAW, encoding="utf8") as archivo:
    item_crudo = json.loads(archivo.readline())  # La primera línea es el primer item

# Las listas (fotos, medios de pago, etc.) se muestran solo con su largo, para que se lea
primeros_campos = {}
for campo, valor in list(item_crudo.items())[:20]:
    if isinstance(valor, list):
        primeros_campos[campo] = f"lista de {len(valor)}"
    else:
        primeros_campos[campo] = valor
primeros_campos

{'seller_address': {'country': {'name': 'Argentina', 'id': 'AR'},
  'state': {'name': 'Buenos Aires', 'id': 'AR-B'},
  'city': {'name': 'Quilmes', 'id': 'TUxBQ1FVSWI1MzY'}},
 'warranty': 'Calidad y mejor precio',
 'sub_status': 'lista de 0',
 'condition': 'new',
 'deal_ids': 'lista de 0',
 'base_price': 15.02,
 'shipping': {'local_pick_up': True,
  'methods': [],
  'tags': [],
  'free_shipping': False,
  'mode': 'not_specified',
  'dimensions': None},
 'non_mercado_pago_payment_methods': 'lista de 3',
 'seller_id': 4086929269,
 'variations': 'lista de 0',
 'site_id': 'MLA',
 'listing_type_id': 'bronze',
 'price': 15.02,
 'attributes': 'lista de 0',
 'buying_mode': 'buy_it_now',
 'tags': 'lista de 1',
 'listing_source': '',
 'parent_item_id': 'MLA8336077177',
 'coverage_areas': 'lista de 0',
 'category_id': 'MLA73721'}

`aplanar_item` lo convierte en una sola fila, sin crear variables nuevas. De las listas se guardan resúmenes, como la cantidad de fotos.

In [3]:
item_aplanado = datos.aplanar_item(item_crudo)
pd.Series(item_aplanado).head(25)

warranty                                          Calidad y mejor precio
condition                                                            new
base_price                                                         15.02
seller_id                                                     4086929269
listing_type_id                                                   bronze
price                                                              15.02
buying_mode                                                   buy_it_now
parent_item_id                                             MLA8336077177
category_id                                                     MLA73721
last_updated                                    2015-10-02T19:19:16.000Z
id                                                         MLA3911152707
official_store_id                                                   None
accepts_mercadopago                                                 True
original_price                                     

## Lo que se descarta

Son columnas constantes o vacías (D02) y las URL, que no aportaron como variables (D70, D80).

In [4]:
datos.COLUMNAS_DESCARTADAS

['site_id',
 'listing_source',
 'international_delivery_mode',
 'coverage_areas',
 'differential_pricing',
 'subtitle',
 'permalink',
 'thumbnail',
 'secure_thumbnail']

## Las tablas

Parsear el JSON es lento, por eso el resultado queda guardado en `data/processed/`.

In [5]:
train, test = datos.cargar_train_test()
print("train", train.shape, "test", test.shape)
print("La única columna que test no tiene es", *sorted(set(train.columns) - set(test.columns)))
train["condition"].value_counts(normalize=True).round(3)

train (70000, 73) test (30000, 72)
La única columna que test no tiene es condition


condition
new     0.537
used    0.463
Name: proportion, dtype: float64

Las clases están bastante balanceadas. Predecir siempre la más frecuente da el accuracy mínimo a superar.